In [1]:
import pandas as pd
import os
import boto3
import yaml
import math
import glob
import numpy as np

# --- rpy2: bridge Python <-> R
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, default_converter
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

from io import StringIO

# Cost benefits
from costs_benefits_ssp.cb_calculate import CostBenefits
from costs_benefits_ssp.model.cb_data_model import TXTable,CostFactor,TransformationCost,StrategyInteraction

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
SCRIPT_DIR_PATH = os.getcwd()
CONFIG_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "config")

In [4]:
def read_yaml(file_path):
    with open(file_path, 'r') as f:
        data = yaml.safe_load(f)
    return data

In [5]:
# Load AWS config
aws_config = read_yaml(os.path.join(CONFIG_DIR_PATH, "aws_credentials_config.yaml"))
PROFILE_NAME = aws_config["profile_name"]
BUCKET_NAME = aws_config["bucket_name"]

# Set your AWS profile
session = boto3.Session(profile_name=PROFILE_NAME)

# Create S3 resource
S3_RESOURCE = session.resource('s3')

# Define run ID and prefix
RUN_ID = "sisepuede_run_2025-10-07t13;30;14.193421"
DIR_ID = 0
RUN_DB_PREFIX = f'run_database/{RUN_ID}/'
MODEL_OUTPUT_PREFIX = f'{RUN_DB_PREFIX}model_output/region=louisiana/model_output_{DIR_ID}/'
MODEL_INPUT_PREFIX = f'{RUN_DB_PREFIX}model_input/region=louisiana/model_input_{DIR_ID}/'
TRANSFER_PREFIX = f"transfers/{RUN_ID}/"

def fetch_csv_from_s3(s3_resource, bucket_name, key):
    obj = s3_resource.Object(bucket_name, key)
    content = obj.get()['Body'].read().decode('utf-8')
    return pd.read_csv(StringIO(content))

output_df = fetch_csv_from_s3(S3_RESOURCE, BUCKET_NAME, f'{MODEL_OUTPUT_PREFIX}data.csv')
input_df = fetch_csv_from_s3(S3_RESOURCE, BUCKET_NAME, f'{MODEL_INPUT_PREFIX}data.csv')
attribute_primary_df = fetch_csv_from_s3(S3_RESOURCE, BUCKET_NAME, f'{TRANSFER_PREFIX}ATTRIBUTE_PRIMARY.csv')
attribute_strategy_df = fetch_csv_from_s3(S3_RESOURCE, BUCKET_NAME, f'{TRANSFER_PREFIX}ATTRIBUTE_STRATEGY.csv')


In [6]:
output_df.head()

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yield_agrc_fruits_tonne,yield_agrc_herbs_and_other_perennial_crops_tonne,yield_agrc_nuts_tonne,yield_agrc_other_annual_tonne,yield_agrc_other_woody_perennial_tonne,yield_agrc_pulses_tonne,yield_agrc_rice_tonne,yield_agrc_sugar_cane_tonne,yield_agrc_tubers_tonne,yield_agrc_vegetables_and_vines_tonne
0,36800368,louisiana,0,0.0,367983.831774,68239.857458,80.234375,79198.541655,6714.566582,1.154590e+06,...,1607.703461,955073.693482,19786.484805,7.132381e+06,0.0,985.642069,2.422231e+06,2.104457e+07,166526.651285,147.841681
1,36800368,louisiana,1,0.0,366050.112647,67881.263666,79.812752,78782.361047,6679.282190,1.148523e+06,...,1634.564583,987915.051061,23080.148287,8.230086e+06,0.0,1159.815534,2.295749e+06,2.036820e+07,172061.964691,151.880248
2,36800368,louisiana,2,0.0,364126.544698,67524.552334,79.393341,78368.365205,6644.183026,1.142487e+06,...,1587.455678,953055.975213,20210.386664,1.270371e+07,0.0,1401.116581,2.304258e+06,2.546716e+07,171681.686103,145.780066
3,36800368,louisiana,3,0.0,362832.342107,67284.552115,79.111156,78089.823191,6620.567832,1.138427e+06,...,1499.071646,959658.852162,22346.292260,6.188630e+06,0.0,1469.617516,2.446406e+06,2.558695e+07,163344.520158,149.114687
4,36800368,louisiana,4,0.0,361132.180759,66969.269871,78.740457,77723.909562,6589.545147,1.133092e+06,...,1593.597169,958934.278998,19717.127162,6.999580e+06,0.0,1132.437039,2.169684e+06,2.372273e+07,167799.254158,164.965084


In [7]:
input_df.head()

,primary_id,region,time_period,area_gnrl_country_ha,area_lndu_infimum_croplands_ha,area_lndu_infimum_flooded_ha,area_lndu_infimum_forests_mangroves_ha,area_lndu_infimum_forests_primary_ha,area_lndu_infimum_forests_secondary_ha,area_lndu_infimum_grasslands_ha,...,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha
0,36800368,louisiana,0,13565900.0,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0,...,12.059233,2.946800,6.177415,0.0,2.755621,8.636027,73.140598,37.547100,28.821448,92.81
1,36800368,louisiana,1,13565900.0,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0,...,12.539800,3.455483,7.165802,0.0,3.259699,8.228317,71.163825,39.000100,29.765171,92.81
2,36800368,louisiana,2,13565900.0,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0,...,12.161233,3.041817,11.119348,0.0,3.958685,8.302446,89.448975,39.119475,28.720595,92.81
3,36800368,louisiana,3,13565900.0,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0,...,12.289167,3.375283,5.436126,0.0,4.167037,8.846059,90.190264,37.352525,29.482348,92.81
4,36800368,louisiana,4,13565900.0,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0,...,12.337700,2.992183,6.177415,0.0,3.226093,7.882382,84.012849,38.551850,32.769776,92.81


In [8]:
attribute_primary_df.head()

,primary_id,design_id,strategy_id,future_id
0,36800368,4,0,0
1,43800438,4,6004,0
2,43800439,4,6004,1
3,43800440,4,6004,2
4,43800441,4,6004,3


In [9]:
primary_ids_run = output_df["primary_id"].unique()
primary_ids_run

array([36800368, 43800438, 43800439, 43800440, 43800441, 43800442,
       43800443, 43800444, 43800445, 43800446, 43800447, 43800448,
       43800449, 43800450, 43800451, 43800452, 43800453, 43800454,
       43800455, 43800456, 43800457, 43800458, 43800459, 43800460,
       43800461, 43800462, 43800463, 43800464, 43800465, 43800466,
       43800467, 43800468, 43800469, 43800470, 43800471, 43800472,
       43800473, 43800474, 43800475, 43800476, 43800477, 43800478,
       43800479, 43800480, 43800481, 43800482, 43800483, 43800484,
       43800485, 43800486, 43800487, 43800488, 43800489, 43800490,
       43800491, 43800492, 43800493, 43800494, 43800495, 43800496,
       43800497, 43800498, 43800499, 43800500, 43800501, 43800502,
       43800503, 43800504, 43800505, 43800506, 43800507, 43800508,
       43800509, 43800510, 43800511, 43800512, 43800513, 43800514,
       43800515, 43800516, 43800517, 43800518, 43800519, 43800520,
       43800521, 43800522, 43800523, 43800524, 43800525, 43800

In [10]:
output_df["primary_id"].nunique()

203

In [11]:
attribute_primary_df[attribute_primary_df["primary_id"].isin(primary_ids_run)]

,primary_id,design_id,strategy_id,future_id
0,36800368,4,0,0
1,43800438,4,6004,0
2,43800439,4,6004,1
3,43800440,4,6004,2
4,43800441,4,6004,3
...,...,...,...,...
200,43800637,4,6004,199
201,43800638,4,6004,200
202,43800639,4,6004,201
203,43800640,4,6004,202


In [12]:
attribute_primary_df[attribute_primary_df.strategy_id == 0]

,primary_id,design_id,strategy_id,future_id
0,36800368,4,0,0


In [13]:
attribute_strategy_df.head()

,strategy_id,strategy_code,strategy,description,transformation_specification,baseline_strategy_id
0,0,BASE,Strategy TX:BASE,NaN,TX:BASE,1
1,1000,AGRC:DEC_CH4_RICE,Singleton - Default Value - AGRC: Improve rice...,NaN,TX:AGRC:DEC_CH4_RICE,0
2,1001,AGRC:DEC_EXPORTS,Singleton - Default Value - AGRC: Decrease Exp...,NaN,TX:AGRC:DEC_EXPORTS,0
3,1002,AGRC:DEC_LOSSES_SUPPLY_CHAIN,Singleton - Default Value - AGRC: Reduce suppl...,NaN,TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN,0
4,1003,AGRC:INC_CONSERVATION_AGRICULTURE,Singleton - Default Value - AGRC: Expand conse...,NaN,TX:AGRC:INC_CONSERVATION_AGRICULTURE,0


### Decomposition

In [ ]:
# --------------------------
# Paths & parameters (edit)
# --------------------------
TARGET_COUNTRY = "LA"

EMISSION_TARGETS_CSV = 'cw/emission_targets_LA_2021.csv'
R_SCRIPT_PATH = 'r_scripts/intertemporal_function_baseline_mapping_timeref.r'
LOCAL_OUTPUT_DIR = 'output_test/'

TIME_PERIOD_REF = 7
S3_DECOMPOSED_DIR_PREFIX = f"{RUN_DB_PREFIX}decomposed_outputs/"

PRIMARY_ID_TO_DECOMPOSE = 43800439 # <- set this per run

In [ ]:
# --------------------------
# Load emission targets (te_all)
# --------------------------
te_all = pd.read_csv(EMISSION_TARGETS_CSV)
cols_needed = ["Subsector", "Gas", "Vars", "Edgar_Class", TARGET_COUNTRY]
missing_cols = set(cols_needed) - set(te_all.columns)
if missing_cols:
    raise ValueError(f"Missing columns in emission targets: {missing_cols}")

te_all = te_all[cols_needed].copy()
te_all = te_all.rename(columns={TARGET_COUNTRY: "tvalue"})
te_all


In [ ]:
# Convert te_all to R data.frame
with localconverter(default_converter + pandas2ri.converter):
    r_te_all = ro.conversion.py2rpy(te_all)

In [ ]:
# --------------------------
# Source the R script & get function
# --------------------------
print(f"Sourcing R script: {R_SCRIPT_PATH}")
ro.r['source'](R_SCRIPT_PATH)
r_rescale = ro.globalenv['rescale']  # function defined in R script

In [ ]:
# --------------------------
# Process a single primary_id
# --------------------------

# 1) Filter to the single primary_id
data_all = output_df.loc[output_df["primary_id"] == PRIMARY_ID_TO_DECOMPOSE].copy()
if data_all.empty:
    raise ValueError(f"No rows found for primary_id={PRIMARY_ID_TO_DECOMPOSE}")

# 2) Basic hygiene
data_all = data_all.fillna(0)

# 3) Validate required columns
for col in ("region", "primary_id", "time_period"):
    if col not in data_all.columns:
        raise ValueError(f"'{col}' column missing in df")

# 4) Sanity: ensure exactly one primary_id
pids = data_all["primary_id"].dropna().unique().tolist()
if len(pids) != 1 or pids[0] != PRIMARY_ID_TO_DECOMPOSE:
    raise ValueError(f"Expected exactly one primary_id={PRIMARY_ID_TO_DECOMPOSE}, got {pids}")

# 5) Regions for this primary_id
rall = data_all["region"].dropna().astype(str).unique().tolist()
if not rall:
    raise ValueError("No regions found after filtering; check 'region' data.")

# 6) Time filter
before = len(data_all)
data_all = data_all.loc[data_all["time_period"] >= TIME_PERIOD_REF].copy()
after = len(data_all)
print(f"[primary_id={PRIMARY_ID_TO_DECOMPOSE}] time_period >= {TIME_PERIOD_REF}: {before} -> {after} rows")
if data_all.empty:
    raise ValueError(f"All rows filtered out for primary_id={PRIMARY_ID_TO_DECOMPOSE} with time_period_ref={TIME_PERIOD_REF}")

# 7) (Optional) sanitize to avoid rpy2 mixed-type issues
def sanitize_for_r(df):
    df = df.copy()
    def _is_scalar(x):
        import pandas as pd
        return not isinstance(x, (list, dict, pd.Series))
    for c in df.columns:
        if not df[c].map(_is_scalar).all():
            df[c] = df[c].astype(str)
        elif df[c].dtype == "object":
            df[c] = df[c].astype("string")
    return df

data_all = sanitize_for_r(data_all)
te_all   = sanitize_for_r(te_all)  # assuming te_all already prepared earlier

# 8) Convert to R (use localconverter, not pandas2ri.activate())

with localconverter(default_converter + pandas2ri.converter):
    r_data_all = ro.conversion.py2rpy(data_all)
with localconverter(default_converter + pandas2ri.converter):
    r_te_all   = ro.conversion.py2rpy(te_all)

# 9) Prepare R args
r_rall       = ro.StrVector([str(x) for x in rall])            # usually length 1
r_init_ids   = ro.StrVector([str(PRIMARY_ID_TO_DECOMPOSE)])                 # exactly one id
r_dir_output = ro.StrVector([LOCAL_OUTPUT_DIR])               # local output dir
r_run        = ro.IntVector([int(PRIMARY_ID_TO_DECOMPOSE)])                 # use primary_id as "run" tag
r_z          = ro.IntVector([1])
r_time_ref   = ro.IntVector([TIME_PERIOD_REF])

print(f"Calling R::rescale(...) for primary_id={PRIMARY_ID_TO_DECOMPOSE} (regions={rall})")
_ = r_rescale(r_z, r_rall, r_data_all, r_te_all, r_init_ids, r_dir_output, r_time_ref, r_run)
print("Finished R::rescale() for single primary_id.")


In [ ]:
# --------------------------
# Read decomposed output and merge with input data
# --------------------------
decomposed_df = pd.read_csv(os.path.join(LOCAL_OUTPUT_DIR, f"louisiana_{PRIMARY_ID_TO_DECOMPOSE}.csv"))

# We need to merge inputs and save it again since we need them for cost-benefit calculations
decomposed_df = pd.merge(decomposed_df, input_df, on=["primary_id", "region", "time_period"], how="left")
decomposed_df.to_csv(os.path.join(LOCAL_OUTPUT_DIR, f"louisiana_{PRIMARY_ID_TO_DECOMPOSE}.csv"), index=False)


decomposed_df.head()

In [ ]:
# --------------------------
# Calculate total emissions
# --------------------------
decomposed_df['total_emissions'] = decomposed_df[[col for col in decomposed_df.columns if col.startswith("emission_co2e_subsector_total")]].sum(axis=1)
df_to_upload = decomposed_df[["primary_id", "time_period", "total_emissions"]]
df_to_upload.head()

In [ ]:
# --------------------------
# Upload to S3
# --------------------------
def upload_df_to_s3(df, s3_resource, bucket, key):
    buffer = StringIO()
    df.to_csv(buffer, index=False)
    s3_resource.Object(bucket, key).put(Body=buffer.getvalue(), ContentType="text/csv")
    print(f"Uploaded to s3://{bucket}/{key}")

# Usage:
s3_key = f"{S3_DECOMPOSED_DIR_PREFIX}emission_total_{PRIMARY_ID_TO_DECOMPOSE}.csv"
upload_df_to_s3(df_to_upload, S3_RESOURCE, BUCKET_NAME, s3_key)


## TODO Cost Benefits

In [ ]:
def run_cba(primary_id_compare, 
            att_primary, 
            att_strategy, 
            BASE_DECOMPOSED_FILE_PATH, 
            DECOMPOSED_FILE_PATH,
            CB_CONFIG_FILE_PATH, 
            PRIMARY_ID_BASE=0):

    # Skip if comparing to itself
    if primary_id_compare == PRIMARY_ID_BASE:
        print(f"Skipping primary_id {primary_id_compare} as it is the baseline.")
        return

    # Set baseline row in attribute primary
    att_primary_copy = att_primary.copy()
    att_primary_copy.iloc[0] = [0, 0, 0, 0] # set first row to baseline values

    # Get the future id if the primary id we're comparing to
    future_id = att_primary.loc[att_primary["primary_id"] == primary_id_compare, "future_id"].values[0]

    # Get the strategy id and code for the primary id we're comparing to
    strategy_id = att_primary.loc[att_primary["primary_id"] == primary_id_compare, "strategy_id"].values[0]
    strategy_code = att_strategy[att_strategy["strategy_id"] == strategy_id]["strategy_code"].values[0]

    print(
        f"\n--- CBA Computation ---\n"
        f"Primary ID      : {primary_id_compare}\n"
        f"Future ID       : {future_id}\n"
        f"Strategy ID     : {strategy_id}\n"
        f"Strategy Code   : {strategy_code}\n"
        f"Base Decomposed : {BASE_DECOMPOSED_FILE_PATH}\n"
        f"Compare Decomp. : {DECOMPOSED_FILE_PATH}\n"
        f"-----------------------\n"
    )


    # Check if decomposed file exists (this is the one belonging to the compare primary id)
    if not os.path.exists(DECOMPOSED_FILE_PATH):
        raise ValueError(f"File {DECOMPOSED_FILE_PATH} does not exist. Skipping.")
    
    # Check if base decomposed file exists (this is the one belonging to the base primary id)
    if not os.path.exists(BASE_DECOMPOSED_FILE_PATH):
        raise ValueError(f"File {BASE_DECOMPOSED_FILE_PATH} does not exist. Skipping.")

    # Load decomposed dfs
    base_decomposed_df = pd.read_csv(BASE_DECOMPOSED_FILE_PATH)
    compare_decomposed_df = pd.read_csv(DECOMPOSED_FILE_PATH)

    # Print shapes
    print(f"Loaded base decomposed data with shape {base_decomposed_df.shape}")
    print(f"Loaded compare decomposed data with shape {compare_decomposed_df.shape}")

    # Set up dataframe that goes into CBA
    ssp_data = pd.concat([base_decomposed_df, compare_decomposed_df]).reset_index(drop=True)
    print(f"Loaded decomposed data with shape {ssp_data.shape}")
    ssp_data["primary_id"] = ssp_data["primary_id"].replace({PRIMARY_ID_BASE: 0})
    ssp_data = ssp_data.replace(np.nan, 0.0)
    strategy_code_base = "BASE"

    # Run CBA
    cb = CostBenefits(ssp_data, att_primary_copy, att_strategy, strategy_code_base)
    cb.ssp_data["future_id"] = 0

    cb.load_cb_parameters(CB_CONFIG_FILE_PATH)

    results_system = cb.compute_system_cost_for_strategy(strategy_code_tx=strategy_code) #NOTE: change this as needed
    results_tx = cb.compute_technical_cost_for_strategy(strategy_code_tx=strategy_code) #NOTE: change this as needed
    results_all = pd.concat([results_system, results_tx], ignore_index=True)

    print(f"Computed raw CBA results with shape {results_all.shape}")

    results_all_pp = cb.cb_process_interactions(results_all)
    results_all_pp_shifted = cb.cb_shift_costs(results_all_pp)

    results_all_pp_shifted["primary_id"] = primary_id_compare
    results_all_pp_shifted["future_id"] = future_id

    # Check it it's empty
    if results_all_pp_shifted.empty:
        print(f"No results for primary_id {primary_id_compare}. Skipping.")

    return results_all_pp_shifted



In [ ]:
attribute_primary_df.head()

In [ ]:
# --------------------------
# Set up CBA paths and parameters
# --------------------------

PRIMARY_ID_BASE = 43800438
BASE_DECOMPOSED_FILE_PATH = os.path.join(LOCAL_OUTPUT_DIR, f"louisiana_{PRIMARY_ID_BASE}.csv")
DECOMPOSED_FILE_PATH = os.path.join(LOCAL_OUTPUT_DIR, f"louisiana_{PRIMARY_ID_TO_DECOMPOSE}.csv")
CB_CONFIG_FILE_PATH = os.path.join(CONFIG_DIR_PATH, "cb_config_params.xlsx")

In [ ]:
# --------------------------
# Run CBA
# --------------------------

cb_raw_df = run_cba(
    primary_id_compare=PRIMARY_ID_TO_DECOMPOSE,
    att_primary=attribute_primary_df,
    att_strategy=attribute_strategy_df,
    BASE_DECOMPOSED_FILE_PATH=BASE_DECOMPOSED_FILE_PATH,
    DECOMPOSED_FILE_PATH=DECOMPOSED_FILE_PATH,
    CB_CONFIG_FILE_PATH=CB_CONFIG_FILE_PATH,
    PRIMARY_ID_BASE=PRIMARY_ID_BASE
)

In [ ]:
# --------------------------
# Post-processing CBA results
# --------------------------

# --- split the 'variable' column into parts ---
# R made 5 columns: name, sector, cb_type, item_1, item_2
# Use n=4 so we get at most 5 pieces even if extra ':' appear later.
parts = cb_raw_df["variable"].astype(str).str.split(":", n=4, expand=True)
parts.columns = ["name", "sector", "cb_type", "item_1", "item_2"]

# append parts to cb_raw_df
cb_data = pd.concat([cb_raw_df, parts], axis=1)

# --- scaling and year ---
cb_data["value"] = cb_data["value"] / 1e9
cb_data["Year"]  = cb_data["time_period"] + 2015

# --- aggregate (sum, skipping NaNs as in na.rm=TRUE) ---
group_cols = ["cb_type", "strategy_code", "primary_id", "future_id", "Year"]
cb_agg = (
    cb_data
    .groupby(group_cols, dropna=False, as_index=False)["value"]
    .sum()
    .rename(columns={"value": "Cumulative"})
)

# --- aggregated format (dcast) ---
agg_cb_df = (
    cb_agg
    .pivot_table(
        index=["primary_id", "future_id", "strategy_code", "Year"],
        columns="cb_type",
        values="Cumulative",
        aggfunc="sum"        # safe even if duplicates appear
        # , fill_value=0     # uncomment if you prefer 0 instead of NaN
    )
    .reset_index()
)

# If you prefer flat columns after pivot (remove the name from columns):
agg_cb_df.columns.name = None
print("unique future_id values:", agg_cb_df["future_id"].nunique())

print(agg_cb_df.columns)

# Keep only relevant columns
agg_cb_df = agg_cb_df[["primary_id","future_id", "strategy_code", "Year","air_pollution", "technical_cost"]]


agg_cb_df.head()


In [ ]:
# --------------------------
# Upload to S3
# --------------------------
S3_CB_DIR_PREFIX = f"{RUN_DB_PREFIX}cb_outputs/"
s3_key = f"{S3_CB_DIR_PREFIX}cb_{PRIMARY_ID_TO_DECOMPOSE}.csv" # This might need to go somewhere else
upload_df_to_s3(agg_cb_df, S3_RESOURCE, BUCKET_NAME, s3_key)


In [ ]:
# --------------------------
# Eliminate local files
# --------------------------
for fname in os.listdir(LOCAL_OUTPUT_DIR):
    if f"louisiana_{PRIMARY_ID_BASE}.csv" not in fname:
        try:
            os.remove(os.path.join(LOCAL_OUTPUT_DIR, fname))
            print(f"Deleted: {fname}")
        except Exception as e:
            print(f"Could not delete {fname}: {e}")